# ablation_run — bảng so sánh cross-encoder cho BÁO CÁO

**Mục đích khác mọi notebook trước: không phải để ăn điểm, mà để có BẢNG.** Báo cáo lần
trước bị hỏi *"đã thử những model nào để chứng minh model đang dùng là tốt nhất?"* — lần
này trả lời bằng số, cùng một giao thức, trên trọn dev300.

### Giao thức — giống hệt nhau cho MỌI model, đó là điều quan trọng nhất

Rổ **fusion top-50** của D · mỗi văn bản lấy **đúng 1 đoạn** hợp câu hỏi nhất
(`pick_chunks k=1`, gộp 1800 ký tự) · chấm **một tầng**, KHÔNG đọc sâu · 300 × 50 =
**15.000 cặp/model**.

Một tầng chứ không hai tầng là **cố ý**: hai tầng có `M`, `K`, `skip` — ba tham số nữa để
cãi nhau, và mỗi model một chi phí khác nhau. Một tầng thì mọi model nhận **đúng cùng
15.000 đoạn văn bản y hệt**, khác biệt duy nhất là bộ chấm. Đó là định nghĩa của ablation.

⚠️ Vì vậy **con số ở đây thấp hơn bài chốt 0.9350** (bài chốt có tầng đọc sâu). Bảng này
dùng để **xếp hạng model với nhau**, không phải để báo cáo điểm hệ thống. Ghi rõ câu đó
trong báo cáo, nếu không sẽ bị hỏi tiếp.

### Chỉ số xuất ra

`recall@1` · **`recall@5`** (chỉ số của cuộc thi) · `recall@10` · `MRR@10`, ở cả
`n_bm25 = 0` (rerank thuần) và `n_bm25 = 1` — đúng khuôn bảng ablation 13/08 để so được
với số cũ. Kèm **số tham số thật** và **giây/cặp**.

### Cột "tham số" là điểm mạnh của báo cáo, đừng bỏ

Ngân sách BTC là **4B cho cả đội, E được 3B**. Một bảng có cột tham số cho thấy nhóm
**chọn model dưới ràng buộc**, không phải chọn bừa. Và nó dựng sẵn chỗ để nói về hai kết
quả đắt giá đã đo hôm 23/08:

- `Qwen3-Reranker-4B` (4,02B) và `8B` (8,19B) **thắng thật** trên tập câu khó (7/15 và
  10/15 so với 6/15) — nhưng **vượt ngân sách, không dùng được**. Đây là phát hiện trung
  thực và hiếm: *chúng tôi biết cái gì tốt hơn, và biết vì sao không được dùng nó.*
- Tập 10 câu mà 8B bắt được **trùng khít** tập 10 câu Gemini 2.5 Flash bắt được — một
  reranker mã nguồn mở tái lập hành vi của LLM thương mại.

### Chi phí — CẮT DANH SÁCH CHO VỪA, đừng chạy hết rồi hết quota giữa chừng

| loại | s/cặp | phút/model |
|---|---|---|
| cross-encoder nhỏ (0,1–0,3B) | ~0,05 | **~13** |
| cross-encoder 0,6B (AITeamVN, bge-v2-m3) | 0,128 | **~32** |
| decoder yes/no 0,6B (Qwen3-Reranker) | ~0,5 | **~125** ⚠️ |

Bảy model encoder ≈ **2,5–3 giờ**. `Qwen3-Reranker-0.6B` một mình đắt hơn cả bảy cái kia
cộng lại — **để nó cuối danh sách**, hết giờ thì bỏ, đã có số dev150 cũ (0.7711) để trích.

**Chạy lại được từ giữa chừng:** mỗi model xong là ghi `scores_ablation_<tag>.json` ngay.
Hết giờ — hoặc gặp lỗi — thì upload thư mục output lên dataset, chạy lại sẽ **bỏ qua model
đã xong**.

### Vì sao chỉ 11 model, trong khi whitelist có ~60 tên

Whitelist gồm cả **embedding / bi-encoder** (`bge-m3`, `multilingual-e5-*`,
`Vietnamese_Embedding`, `Qwen3-Embedding-*`, `vietnamese-bi-encoder`…) — đó là **khâu truy
hồi của D**, không phải bộ chấm của E — và cả **LLM sinh văn bản** (`Vi-Qwen2-*-RAG`,
`PhoGPT-4B`, `Qwen3.5-*`, `ViLegalQwen*`…), vốn không dùng làm cross-encoder được.
Lọc ra thì **chỉ có 13 cái là reranker**, và bảng này chạy 11 trong số đó.

Hai cái còn lại, và lý do bỏ:
- `infgrad/Prism-Qwen3.5-Reranker-2B` — decoder 2B, ước **~6 giờ cho một dòng**. Quá đắt.
- `jinaai/jina-reranker-v3.5` — **listwise**, xếp cả rổ cùng lúc, API khác hẳn nên không
  dùng chung giao thức pointwise này được. Muốn có thì phải đo riêng, ghi rõ là khác cách.

Hai dòng thêm vào có chủ ý:
- `ms-marco-MiniLM-L6-v2` là reranker **tiếng Anh thuần** → **đối chứng âm**. Nó phải tệ
  rõ rệt; nếu không thì chính phép đo có vấn đề.
- `thanhtantran/Vietnamese_Reranker` — **đã tra 23/08: KHÔNG phải bản sao.** Cùng nền
  `bge-reranker-v2-m3`, cùng ~1,1 triệu bộ ba, nhưng **nhóm tác giả khác** (Nguyễn Nho
  Trung, Nguyễn Nhật Quang, Nguyễn Văn Huy). Tức là **hai đội độc lập fine-tune cùng một
  model nền cho cùng một việc** — so hai cái với nhau là một dòng rất có giá trị.

### ⚠️ `max_length` phải khai theo TỪNG MODEL, không suy từ "có tách từ hay không"

23/08: `PhoRanker` chết bằng **device-side assert** vì bị đưa `max_length=512`, trong khi
**PhoBERT-base chỉ có `max_position_embeddings = 258`** — quá 256 token là tra bảng vị trí
ngoài biên. CLAUDE.md đã ghi "PhoRanker/ViRanker: PhoBERT, trần 256 token" từ lâu mà tôi
vẫn đặt 512 vì suy nhầm từ cờ `need_seg`. Giờ mỗi model tự khai, mặc định 1024.

Lưu ý cho báo cáo: **`PhoRanker` chỉ đọc 256 token, các model khác đọc 1024.** Không phải
lỗi giao thức mà là **trần cứng của kiến trúc** — phải ghi rõ trong chú thích bảng.
`ViRanker` thì KHÔNG dính: nó **567.755.777 tham số = nền XLM-R**, không phải PhoBERT như
tên gọi gợi ý; nó chạy `max_length=512` trót lọt, và với đoạn 900 ký tự (~300 token) thì
512 **không hề cắt**, nên con số của nó vẫn dùng được, khỏi chạy lại.

### ⚠️ Hai model dùng `trust_remote_code` nằm CUỐI DANH SÁCH, có lý do

Lượt 23/08 `gte-multilingual-reranker-base` bắn **device-side assert** (`index out of
bounds` trong tra bảng nhúng, rồi `CUBLAS_STATUS_NOT_SUPPORTED`) trên T4. Lỗi loại này
**làm hỏng luôn CUDA context của cả tiến trình** — không cứu được, mọi model sau đó đều
chết theo, và ngay cả `torch.cuda.empty_cache()` trong khối `finally` cũng ném lỗi, khiến
`try/except` quanh từng model trở nên vô dụng. Đã sửa hai chỗ: khối dọn dẹp tự bọc `try`,
và sau mỗi model có kiểm **CUDA còn sống không** — chết thì **dừng sạch**, giữ nguyên kết
quả đã chấm, để chạy lại là tiếp được.

**Upload:** không cần gì mới. Internet ON (tải model).

### 💡 Tiết kiệm 32 phút: hàng `AITeamVN` ĐÃ CÓ SẴN

`Ketqua_E/scores_dev300_enrich_plain.json` chính là model này trên **đúng giao thức này**
(đã đối chiếu: R@5 khớp 0.7467 / 0.8267 / 0.8700 / 0.8900 ở cả bốn `n`). Đổi tên thành
`scores_ablation_AITeamVN.json`, upload lên dataset, notebook sẽ bỏ qua nó. Bước nạp nhận
cả hai định dạng nên khỏi sửa file.


In [ ]:
!pip install -q -U sentence-transformers underthesea

In [ ]:
# ===== Bước 1: dựng 15.000 cặp — DÙNG CHUNG cho mọi model =====
import os, sys, json, time, gc, glob, traceback
import torch
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print(f"thiết bị: {DEV}" + (f" · {torch.cuda.get_device_name(0)}" if DEV == "cuda" else ""))

# KHÁC oracle13: lượt này 15.000 cặp × 9 model. Trên CPU là ~3,5 GIỜ MỖI MODEL = hơn 30h.
# Không chặn thì nó cứ thế chạy cả đêm rồi hết phiên mà chưa xong model thứ hai.
CHO_PHEP_CPU = False
assert DEV == "cuda" or CHO_PHEP_CPU, (
    "CẦN GPU. 15.000 cặp/model trên CPU ~3,5h mỗi model. "
    "Settings -> Accelerator -> GPU. (Cố tình chạy CPU thì đặt CHO_PHEP_CPU = True.)")

EXC = 900                                   # ký tự trích mỗi văn bản, y hệt hard15/enrich
INPUT_DIR = next(p for p in ("/kaggle/input/project-ir",
                             "/kaggle/input/datasets/locdovan211/project-ir")
                 if os.path.isdir(p))
CTX_DIR = next(p for p in (f"{INPUT_DIR}/selected-contexts/selected-contexts",
                           f"{INPUT_DIR}/selected-contexts")
               if os.path.isdir(p) and any(f.startswith("context_") for f in os.listdir(p)))
OUT = "/kaggle/working/outputs"; os.makedirs(OUT, exist_ok=True)
sys.path.append(INPUT_DIR)

import deep_chunk as DC
from rerank import load_reranker
from rerank_from_d import blend_bm25_first
DC.MERGE_CHARS = 1800

dev  = json.load(open(f"{INPUT_DIR}/dev_300_locked.json", encoding="utf-8"))
cand = json.load(open(f"{INPUT_DIR}/fusion_rrf_top50_dev_1000.json", encoding="utf-8"))
qids = [q for q in dev if q in cand]
GOLD = {q: {str(a) for a in dev[q]["answer"]} for q in qids}
ORDER= {q: [str(c["doc_id"]) for c in
            sorted(cand[q], key=lambda c: -float(c["rrf_score"]))] for q in qids}

t0 = time.time()
index, texts = [], []
for i, q in enumerate(qids, 1):
    qs = dev[q]["question"]
    for d in ORDER[q]:
        e = DC.pick_chunks(qs, CTX_DIR, d, k=1)
        index.append((q, d)); texts.append((e[0] if e else "")[:EXC])
    if i % 100 == 0: print(f"  băm {i}/{len(qids)} | {(time.time()-t0)/60:.1f} phút", flush=True)
QTEXT = [dev[q]["question"] for q, _ in index]
print(f"\n{len(qids)} câu × 50 = {len(index):,} cặp/model")
print(f"TRẦN rổ 50: {sum(len(GOLD[q] & set(ORDER[q]))/len(GOLD[q]) for q in qids)/len(qids):.4f}")

In [ ]:
# ===== Bước 2: chỉ số + tách từ (cho họ PhoBERT) =====
def do(per, n):
    """recall@1/5/10 + MRR@10 cho một bảng điểm {qid: {doc: score}}."""
    r = {1: 0.0, 5: 0.0, 10: 0.0}; mrr = 0.0
    for q in qids:
        rk = sorted(per[q], key=lambda d: -per[q][d])
        p10 = blend_bm25_first(rk, ORDER[q], k=10, n_bm25=n)
        for k in r: r[k] += len(GOLD[q] & set(p10[:k])) / len(GOLD[q])
        hit = next((i for i, d in enumerate(p10, 1) if d in GOLD[q]), 0)
        mrr += 1.0 / hit if hit else 0.0
    m = len(qids)
    return {"R@1": r[1]/m, "R@5": r[5]/m, "R@10": r[10]/m, "MRR@10": mrr/m}

def gioi_han(mid, want, trc=False):
    """Hạ max_length xuống trần THẬT của kiến trúc, đọc từ config trước khi tải trọng số.

    Đây là chỗ đã giết `PhoRanker` (PhoBERT `max_position_embeddings`=258 mà bị đưa 512):
    quá trần thì tra bảng vị trí ngoài biên -> device-side assert, hỏng CUDA context.
    Bảng tra tay thì sớm muộn cũng sót một model; đọc thẳng từ config thì không.
    `ms-marco-MiniLM` (BERT tiếng Anh, trần 512) đặc biệt nguy: tokenizer tiếng Anh băm
    tiếng Việt có dấu ra RẤT nhiều mảnh, 900 ký tự có thể vọt quá 512 token.
    """
    try:
        from transformers import AutoConfig
        lim = getattr(AutoConfig.from_pretrained(mid, trust_remote_code=trc),
                      "max_position_embeddings", None)
        if lim and lim < 10_000:
            want = min(want, lim - 2)
    except Exception as e:
        print(f"    (không đọc được config {mid}: {type(e).__name__}) — giữ {want}")
    return want

def don_cache():
    """11 model ~15GB. Kaggle chỉ có ~20GB đĩa ghi -> dọn sau mỗi model, đừng để hết chỗ."""
    import shutil
    for p in ("/root/.cache/huggingface/hub", "/root/.cache/torch/sentence_transformers"):
        shutil.rmtree(p, ignore_errors=True)

try:                                        # PhoBERT cần văn bản ĐÃ TÁCH TỪ, không thì điểm rác
    from functools import lru_cache
    from underthesea import word_tokenize
    # nhớ kết quả: 300 câu hỏi nhưng bị lặp 50 lần mỗi câu -> không cache là tách thừa 299/300
    seg = lru_cache(maxsize=None)(lambda s: word_tokenize(s, format="text"))
    print("underthesea OK — chạy được nhánh PhoBERT")
except Exception as e:
    seg = None
    print(f"KHÔNG có underthesea ({e}) — bỏ qua PhoRanker/ViRanker")

In [ ]:
# ===== Bước 3: chạy từng model. Hỏng một cái KHÔNG giết cả bảng =====
# (tag, model_id, kwargs, cần_tách_từ)  — XẾP THEO CHI PHÍ TĂNG DẦN, hết giờ thì cắt đuôi
# Kiến trúc CHUẨN trước (an toàn), `trust_remote_code` XUỐNG CUỐI: 23/08 `gte-multi` bắn
# device-side assert làm hỏng luôn CUDA context, mọi model sau nó đều chết theo.
MODELS = [
 ("mMiniLMv2",  "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",     {}, False),
 ("msmarcoL6",  "cross-encoder/ms-marco-MiniLM-L6-v2",            {}, False),   # tiếng ANH
 ("bge-base",   "BAAI/bge-reranker-base",                         {}, False),
 ("thanhtan",   "thanhtantran/Vietnamese_Reranker",               {}, False),   # đội khác, cùng nền
 ("ViRanker",   "namdp-ptit/ViRanker",                            {}, True),   # nền XLM-R, 1024 OK
 ("PhoRanker",  "itdainb/PhoRanker",                              {}, True),   # gioi_han -> 256
 ("bge-v2-m3",  "BAAI/bge-reranker-v2-m3",                        {}, False),
 ("AITeamVN",   "AITeamVN/Vietnamese_Reranker",                   {}, False),   # đang dùng
 ("qwen3rr06",  "Qwen/Qwen3-Reranker-0.6B",                       {}, False),   # ĐẮT (~2h)
 # --- từ đây là mã tuỳ biến, có tiền sử làm sập CUDA context. Để CUỐI CÙNG. ---
 ("jina-v2",    "jinaai/jina-reranker-v2-base-multilingual",
                                        dict(trust_remote_code=True), False),
 ("gte-multi",  "Alibaba-NLP/gte-multilingual-reranker-base",
                                        dict(trust_remote_code=True), False),
]

def cuda_con_song():
    """Một phép tính bé xíu. Sau device-side assert thì nó ném lỗi -> biết context đã chết."""
    if DEV != "cuda":
        return True
    try:
        torch.zeros(1, device="cuda").add_(1); torch.cuda.synchronize(); return True
    except Exception:
        return False

T0 = time.time()
RES = {}
for tag, mid, kw, need_seg in MODELS:
    if time.time() - T0 > 10 * 3600:        # trần 12h/lượt — dừng trước khi bị cắt ngang
        print("\n!! Đã chạy 10 giờ, dừng để kịp lưu. Upload outputs/ rồi chạy tiếp."); break
    p = f"{OUT}/scores_ablation_{tag}.json"
    # Tìm ĐỆ QUY trong dataset: upload cả thư mục `outputs/` thì file nằm ở
    # {INPUT_DIR}/outputs/..., tìm phẳng sẽ không thấy và chấm lại từ đầu.
    old = glob.glob(f"{INPUT_DIR}/**/scores_ablation_{tag}.json", recursive=True)
    src = p if os.path.isfile(p) else (old[0] if old else None)
    if src:
        mp = os.path.join(os.path.dirname(src), f"meta_ablation_{tag}.json")
        mt = json.load(open(mp, encoding="utf-8")) if os.path.isfile(mp) else {}
        raw = json.load(open(src, encoding="utf-8"))
        # nhận cả hai dạng: {doc: điểm} và {doc: {"ce":..,"bm25":..}} của enrich_run
        RES[tag] = ({q: {d: (v["ce"] if isinstance(v, dict) else v) for d, v in e.items()}
                     for q, e in raw.items()}, mid, mt)
        print(f"{tag}: đã có, bỏ qua"); continue
    if need_seg and seg is None:
        print(f"{tag}: BỎ — cần tách từ mà không có underthesea"); continue
    try:
        t0 = time.time()
        kw["max_length"] = gioi_han(mid, kw.get("max_length", 1024),
                                    kw.get("trust_remote_code", False))
        print(f"[{tag}] max_length = {kw['max_length']}", flush=True)
        m = load_reranker(mid, device=DEV, **kw)
        pr = [[seg(q), seg(t)] for q, t in zip(QTEXT, texts)] if need_seg else \
             [[q, t] for q, t in zip(QTEXT, texts)]
        sc = m.predict(pr)
        el = time.time() - t0
        per = {}
        for (q, d), v in zip(index, sc): per.setdefault(q, {})[d] = float(v)
        # đếm tham số TẠI CHỖ, đừng chép tay từ model card — đã có tiền lệ tên "4B" mà thật 4,02B
        mt = {"params": sum(x.numel() for x in m.model.parameters()),
              "sec_per_pair": el / len(index)}
        json.dump(per, open(p, "w", encoding="utf-8"), ensure_ascii=False)
        json.dump(mt, open(f"{OUT}/meta_ablation_{tag}.json", "w", encoding="utf-8"))
        RES[tag] = (per, mid, mt)
        print(f">>> {tag}: {el/60:.1f} phút · {mt['sec_per_pair']:.3f} s/cặp · "
              f"{mt['params']/1e9:.3f}B tham số -> đã lưu"
              f"   [tổng {(time.time()-T0)/60:.0f} phút]", flush=True)
    except Exception:
        print(f">>> {tag}: HỎNG\n{traceback.format_exc()[-500:]}")
    finally:
        # BẮT BUỘC bọc try: sau device-side assert thì CHÍNH empty_cache() cũng ném lỗi, mà
        # nó nằm NGOÀI except ở trên -> văng khỏi vòng lặp, giết cả bảng. Đã dính 23/08.
        globals().pop("m", None); globals().pop("pr", None)
        don_cache()
        try:
            gc.collect()
            if DEV == "cuda":
                torch.cuda.empty_cache()
        except Exception as e:
            print(f"    (dọn dẹp lỗi, bỏ qua: {type(e).__name__})")
    if not cuda_con_song():
        print("\n!! CUDA CONTEXT ĐÃ HỎNG (device-side assert). Mọi model sau đều chết theo —")
        print("   lỗi loại này không cứu được trong cùng tiến trình. DỪNG SẠCH tại đây.")
        print(f"   {len(RES)} model đã chấm vẫn nằm nguyên trong {OUT}/.")
        print("   Tải về, upload lên dataset, chạy lại -> nó BỎ QUA những cái đã xong.")
        break
print(f"\nxong {len(RES)}/{len(MODELS)} model — TẢI {OUT}/ VỀ TRƯỚC KHI ĐÓNG PHIÊN")

In [ ]:
# ===== Bước 4: in bảng Markdown, dán thẳng vào báo cáo =====
rows = []
for tag, (per, mid, mt) in RES.items():
    a, b = do(per, 0), do(per, 1)
    rows.append((a["R@5"], tag, mid, mt.get("params"), a, b, mt.get("sec_per_pair")))
rows.sort(reverse=True)                      # xếp theo n=0: bộ chấm THUẦN, không pha rổ

f2 = lambda v, f: (f % v) if v is not None else "—"
print("| Model | Tham số | R@1 | **R@5 (thuần)** | R@10 | MRR@10 | R@5 (+RRF) | s/cặp |")
print("|---|---|---|---|---|---|---|---|")
for r5, tag, mid, pa, a, b, sp in rows:
    star = " ← đang dùng" if tag == "AITeamVN" else ""
    print(f"| `{mid}`{star} | {f2(pa and pa/1e9, '%.3fB')} | {a['R@1']:.4f} | "
          f"**{a['R@5']:.4f}** | {a['R@10']:.4f} | {a['MRR@10']:.4f} | "
          f"{b['R@5']:.4f} | {f2(sp, '%.3f')} |")

print("\n> Giao thức: dev300 · rổ fusion top-50 của D · 1 đoạn/văn bản (BM25 mức đoạn,")
print("> gộp 1800 ký tự) · chấm MỘT tầng, không đọc sâu · 15.000 cặp mỗi model · n_bm25=1.")
print("> Con số THẤP HƠN hệ thống hoàn chỉnh (0.9350) vì bảng này bỏ tầng đọc sâu —")
print("> nó dùng để xếp hạng bộ chấm với nhau, không phải để báo cáo điểm hệ thống.")
if "AITeamVN" in RES:
    top = rows[0]
    print(f"\nDẪN ĐẦU: {top[1]}  R@5 = {top[0]:.4f}")
    if top[1] != "AITeamVN":
        me = next(r for r in rows if r[1] == "AITeamVN")
        print(f"!! AITeamVN chỉ {me[0]:.4f} — KÉM {(top[0]-me[0])*100:.2f} điểm. "
              f"Nếu chênh ≥2,0 thì đây là cần gạt thật, không chỉ là dòng cho báo cáo.")